# Hydrophone Calibration Analysis
This notebook loads the 20 hydrophone recordings, aligns them perfectly in time using cross-correlation, and calculates their Power Spectral Density (PSD) for direct sensitivity comparison.

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import welch, correlate

In [ ]:
def load_and_align_signals(input_dir="recorded_wavs"):
    wav_pattern = os.path.join(input_dir, "hydrophone_*.wav")
    wav_files = glob.glob(wav_pattern)
    
    if not wav_files:
        print(f"No files found in {input_dir}. Please add .wav files.")
        return [], [], 0

    def extract_number(filename):
        nums = ''.join(filter(str.isdigit, os.path.basename(filename)))
        return int(nums) if nums else 0
        
    wav_files.sort(key=extract_number)
    
    signals = []
    names = []
    sample_rates = []
    
    for wav_file in wav_files:
        filename = os.path.basename(wav_file)
        hydrophone_id = filename.split('.')[0]
        
        sr, data = wavfile.read(wav_file)
        if len(data.shape) > 1:
            data = data[:, 0]
            
        if data.dtype == np.int16:
            data = data.astype(np.float32) / np.iinfo(np.int16).max
        elif data.dtype == np.int32:
            data = data.astype(np.float32) / np.iinfo(np.int32).max
        elif data.dtype == np.float32 or data.dtype == np.float64:
            data = data.astype(np.float32)
            
        signals.append(data)
        names.append(hydrophone_id)
        sample_rates.append(sr)
        
    print(f"Loaded {len(signals)} hydrophone recordings.")
    
    # Align signals using cross-correlation against the first signal
    ref_signal = signals[0]
    aligned_signals = [ref_signal]
    
    for i in range(1, len(signals)):
        sig = signals[i]
        
        # Use FFT correlation for speed on large arrays
        correlation = correlate(ref_signal, sig, mode='full', method='fft')
        lag = np.argmax(correlation) - (len(sig) - 1)
        
        print(f"Aligning {names[i]}: offset = {lag} samples")
        
        if lag > 0:
            # sig is delayed relative to ref_signal
            aligned_sig = np.pad(sig, (lag, 0), mode='constant')[:len(ref_signal)]
        elif lag < 0:
            # sig is ahead of ref_signal
            aligned_sig = sig[-lag:]
            if len(aligned_sig) < len(ref_signal):
                aligned_sig = np.pad(aligned_sig, (0, len(ref_signal) - len(aligned_sig)), mode='constant')
            else:
                aligned_sig = aligned_sig[:len(ref_signal)]
        else:
            aligned_sig = sig[:len(ref_signal)]
            
        aligned_signals.append(aligned_sig)
        
    return aligned_signals, names, sample_rates[0]

In [ ]:
# Load and align the signals
aligned_signals, names, sr = load_and_align_signals("recorded_wavs")

if aligned_signals:
    plt.figure(figsize=(14, 8))
    
    for sig, name in zip(aligned_signals, names):
        # Calculate Welch PSD
        f, Pxx = welch(sig, fs=sr, nperseg=8192, scaling='density')
        Pxx_dB = 10 * np.log10(Pxx + 1e-12)
        
        plt.semilogx(f, Pxx_dB, label=name, alpha=0.8, linewidth=1.5)
        
    plt.title("Hydrophone Calibration Comparison (Time-Aligned Sweep Response)")
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("Power Spectral Density [dB/Hz]")
    plt.xlim(100, 10000)
    plt.grid(True, which="both", ls="--", alpha=0.5)
    
    # Configure legend
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small', ncol=2 if len(names)>15 else 1)
    plt.tight_layout()
    
    # Save and display
    plt.savefig("calibration_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()